In [ ]:
# ========== 安装依赖：行情数据 + 表格 + Gradio UI + 数值计算 ==========
# python -m pip：用当前解释器对应的 pip，避免装错环境

# yfinance：拉 Yahoo Finance 行情；pandas/numpy：数据处理；gradio：界面
!python -m pip install yfinance pandas gradio numpy


In [ ]:
# ========== 导入：装好依赖后再跑本格 ==========

# gradio：搭「买卖信号」Web UI
import gradio as gr
# pandas：表格/时序处理（常与 yfinance 结果配合）
import pandas as pd
# yfinance：按 ticker 拉历史行情
import yfinance as yf
# numpy：数值计算（均线等）
import numpy as np

# 自检：导入成功就会打印这句（emoji 与原文保持一致）
print("✅ Imports successful!")


In [ ]:
# ========== Gradio UI：热门 ticker 下拉 + 手动输入 + 周期选择 ==========
# 说明：本格调用 get_signals / reset_portfolio；若笔记本未定义这两函数，需补齐后再 launch

# Blocks + Soft 主题；title 显示在浏览器标签
with gr.Blocks(title="TRADING CODE", theme=gr.themes.Soft()) as demo:
    # 页头：均线交叉买卖规则说明（UI 文案字符串保持原样）
    gr.Markdown("""
    # 买或卖！
    - **BUY** when 5-day MA > 20-day MA
    - **SELL** when 5-day MA < 20-day MA
    - Select from popular tickers or type your own
    """)
    
    # 左右两列：左控件、右输出
    with gr.Row():
        with gr.Column():
            # 热门股票代码下拉；默认 AAPL
            popular_tickers = gr.Dropdown(
                label="Popular Tickers",
                choices=[
                    "AAPL", "MSFT", "GOOG", "TSLA", "AMZN", "META", "NVDA", 
                    "MARA", "COIN", "AMD", "NFLX", "PLTR", "MSTR", "RIOT",
                    "BAC", "JPM", "F", "GM", "DIS", "PYPL", "UBER"
                ],
                value="AAPL"
            )
            
            # 可选：手动输入任意 ticker，非空时优先生效
            manual_ticker = gr.Textbox(
                label="Or Enter Custom Ticker",
                placeholder="Type any ticker..."
            )
            
            # 拉取历史的时间窗口：1 周 / 1 月 / 3 月 / 6 月
            period = gr.Dropdown(
                ["1wk", "1mo", "3mo", "6mo"], 
                label="Period", 
                value="1mo"
            )
            
            # TRADE NOW 触发信号；Reset 清空虚拟持仓（依赖 reset_portfolio）
            with gr.Row():
                get_btn = gr.Button("🚀 TRADE NOW", variant="primary", scale=2)
                reset_btn = gr.Button("🔄 Reset Portfolio", scale=1)
        
        with gr.Column():
            # 右侧用 Markdown 展示信号/组合结果
            out = gr.Markdown()
    
    # 解析最终 ticker：手动框有内容就用手动，否则用下拉
    def get_ticker_from_input(popular, manual):
        return manual if manual.strip() != "" else popular
    
    # 按钮回调：拼出 ticker 后交给 get_signals（需在其他单元定义）
    def trade_with_input(popular, manual, period):
        ticker = get_ticker_from_input(popular, manual)
        return get_signals(ticker, period)
    
    # 绑定点击事件：交易 / 重置
    get_btn.click(trade_with_input, [popular_tickers, manual_ticker, period], out)
    reset_btn.click(reset_portfolio, None, out)
    
    # 页脚使用说明（英文步骤原文保留）
    gr.Markdown("""
    ---
    # 它是如何运作的
    1. Select a popular ticker from dropdown OR type any ticker
    2. Choose time period
    3. Click TRADE NOW - system will BUY or SELL based on MA crossover
    4. Track your virtual portfolio in real-time
    """)

# share=True：尝试生成公网临时链接，方便演示
demo.launch(share=True)
